# 05 — Joint-State Kalman Filter (Updated filenames)


In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(9423)

FIG_DIR = "figures/joint_state_kalman"
RESULTS_DIR = "results"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

def save_fig(name):
    plt.savefig(f"{FIG_DIR}/05_joint_state_{name}.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"{FIG_DIR}/05_joint_state_{name}.pdf", bbox_inches="tight")


In [ ]:
def rabi_model(t, A, Omega, B):
    return A*np.sin(0.5*Omega*t)**2 + B

def rmse(a,b): return float(np.sqrt(np.mean((a-b)**2)))

def moving_average(x,w=4):
    y=np.zeros_like(x)
    for i in range(len(x)):
        y[i]=np.mean(x[max(0,i-w+1):i+1])
    return y

def kalman_1d(z,q,r):
    x=z[0]; p=1
    out=[]
    for m in z:
        p+=q
        k=p/(p+r)
        x=x+k*(m-x)
        p=(1-k)*p
        out.append(x)
    return np.array(out)

def kalman_joint(z,q_om,q_b,q_cross,r_om,r_b):
    F=np.eye(2); H=np.eye(2)
    Q=np.array([[q_om,q_cross],[q_cross,q_b]])
    R=np.array([[r_om,0],[0,r_b]])
    x=z[0].reshape(2,1); P=np.eye(2)
    out=[]
    for i in range(len(z)):
        x=F@x; P=F@P@F.T+Q
        y=z[i].reshape(2,1)-H@x
        S=H@P@H.T+R
        K=P@H.T@np.linalg.inv(S)
        x=x+K@y
        P=(np.eye(2)-K@H)@P
        out.append(x.ravel())
    return np.array(out)


In [ ]:
# synthetic
n=48
t_blocks=np.arange(n)
pulse_t=np.linspace(0,10,140)

A=0.88; Om=2.45; B=0.06

shared=0.018*np.sin(2*np.pi*t_blocks/23+0.25)
omega_private=0.05*np.sin(2*np.pi*t_blocks/21)+0.014*np.sin(2*np.pi*t_blocks/9)
b_private=0.0013*t_blocks+0.005*np.sin(2*np.pi*t_blocks/14)

dOm=omega_private+shared
dB=b_private+0.22*shared

Om_u=Om+dOm; B_u=B+dB

# simulate + fit
Y=[]
for k in range(n):
    y=rabi_model(pulse_t,A,Om_u[k],B_u[k])
    y+=0.04*np.random.randn(len(pulse_t))
    Y.append(np.clip(y,0,1))
Y=np.array(Y)

fit=[]
for k in range(n):
    p,_=curve_fit(rabi_model,pulse_t,Y[k],[0.85,2.4,0.05],bounds=([0,0,0],[1.2,5,0.3]),maxfev=20000)
    fit.append(p)
fit=np.array(fit)

dOm_est=fit[:,1]-Om
dB_est=fit[:,2]-B


In [ ]:
# estimators
dOm_ma=moving_average(dOm_est)
dB_ma=moving_average(dB_est)

r_om=np.var(dOm_est-dOm)
r_b=np.var(dB_est-dB)

q_om=8e-4; q_b=1e-5

dOm_k=kalman_1d(dOm_est,q_om,r_om)
dB_k=kalman_1d(dB_est,q_b,r_b)

z=np.column_stack([dOm_est,dB_est])
q_cross=0.2*np.sqrt(q_om*q_b)

joint=kalman_joint(z,q_om,q_b,q_cross,r_om,r_b)
dOm_j=joint[:,0]; dB_j=joint[:,1]


In [ ]:
def eval_policy(dOm_hat,dB_hat):
    Om_c=Om+dOm-dOm_hat
    B_c=B+dB-dB_hat
    tgt=rabi_model(pulse_t,A,Om,B)
    rms=[]
    for k in range(n):
        y=rabi_model(pulse_t,A,Om_c[k],B_c[k])
        rms.append(rmse(y,tgt))
    return np.array(rms)

policies={
    "none":eval_policy(np.zeros(n),np.zeros(n)),
    "moving_avg":eval_policy(dOm_ma,dB_ma),
    "scalar":eval_policy(dOm_k,dB_k),
    "joint":eval_policy(dOm_j,dB_j)
}


In [ ]:
# plots
plt.figure()
for k,v in policies.items(): plt.plot(v,label=k)
plt.legend(); plt.title("RMSE comparison")
save_fig("response_rmse_comparison")
plt.show()

plt.figure()
plt.plot(dOm,label="true")
plt.plot(dOm_est,label="meas")
plt.plot(dOm_k,label="scalar")
plt.plot(dOm_j,label="joint")
plt.legend(); plt.title("Omega drift")
save_fig("omega_estimator_comparison")
plt.show()

plt.figure()
plt.plot(dB,label="true")
plt.plot(dB_est,label="meas")
plt.plot(dB_k,label="scalar")
plt.plot(dB_j,label="joint")
plt.legend(); plt.title("Offset drift")
save_fig("offset_estimator_comparison")
plt.show()


In [ ]:
summary={k:float(np.mean(v)) for k,v in policies.items()}
with open(f"{RESULTS_DIR}/joint_state_kalman_summary.json","w") as f:
    json.dump(summary,f,indent=2)
summary
